In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns
import math
from cmdstanpy import CmdStanModel

/opt/anaconda3/envs/stan_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
snp = pd.read_json('../json_snp.json')
snp['d_mean'] = snp['d_mean']*100
snp['d_var'] = snp['d_var']*100
print(snp)

    group    d_mean     d_var  adjusted_factor
0  [0, 0]  0.817305  0.023009          0.18755
1  [0, 1] -0.010856  0.016651          0.18755
2  [0, 2] -0.090904  0.019700          0.18755
3  [0, 3] -0.715544  0.027234          0.18755
4  [1, 1]  0.679156  0.021560          0.18755
5  [1, 2]  0.140838  0.017736          0.18755
6  [1, 3] -0.809138  0.030801          0.18755
7  [2, 2]  0.804949  0.023561          0.18755
8  [2, 3] -0.854883  0.029888          0.18755
9  [3, 3]  2.379564  0.062573          0.18755


In [4]:
json_snp = snp.to_dict(orient = 'list')
json_snp['N_obs'] = len(snp)
json_snp['N_pop'] = 4
json_snp['ancestral_T'] = 800

In [8]:
snp_model = CmdStanModel(stan_file="snp_alone_with_weights.stan")

16:48:40 - cmdstanpy - INFO - compiling stan file /Users/qi/Documents/GitHub/admix_stan/new_pipeline/snp_alone_with_weights.stan to exe file /Users/qi/Documents/GitHub/admix_stan/new_pipeline/snp_alone_with_weights
16:48:45 - cmdstanpy - INFO - compiled model executable: /Users/qi/Documents/GitHub/admix_stan/new_pipeline/snp_alone_with_weights


In [9]:
fit = snp_model.sample(
    data=json_snp,
    chains=4, parallel_chains=4,
    iter_warmup=1500, iter_sampling=1500,
    adapt_delta=0.99,        # 0.95 → 0.99 (or 0.999 if needed)
    max_treedepth=12,        # bump to 15 if “max treedepth” saturates
    inits=0,                 # calm initial values
    # step_size=0.5,         # optional: can help if still divergent
)

16:48:50 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/3000 [00:00<?, ?it/s, (Warmup)]





chain 1:   7%|▋         | 200/3000 [00:00<00:03, 804.17it/s, (Warmup)]


chain 1:  10%|█         | 300/3000 [00:00<00:03, 823.59it/s, (Warmup)]



chain 1:  17%|█▋        | 500/3000 [00:00<00:03, 827.10it/s, (Warmup)]


chain 1:  20%|██        | 600/3000 [00:00<00:02, 863.15it/s, (Warmup)]


chain 1:  27%|██▋       | 800/3000 [00:00<00:02, 987.74it/s, (Warmup)]


chain 1:  33%|███▎      | 1000/3000 [00:01<00:01, 1047.35it/s, (Warmup)]



chain 1:  40%|████      | 1200/3000 [00:01<00:01, 1077.52it/s, (Warmup)]




chain 1:  50%|█████     | 1500/3000 [00:01<00:01, 1067.62it/s, (Sampling)]


chain 1:  53%|█████▎    | 1600/3000 [00:01<00:01, 930.41it/s, (Sampling)] 


chain 1:  57%|█████▋    | 1700/3000 [00:01<00:01, 910.74it/s, (Sampling)]


chain 1:  60%|██████    | 1800/3000 [00:01<00:01, 902.68it/s, (Sampling)]


chain 1:  67%|██████▋   | 2000/3000 [00:02<00:01, 874.8


16:48:53 - cmdstanpy - INFO - CmdStan done processing.


In [10]:
fit.summary()

,Mean,MCSE,StdDev,MAD,5%,50%,95%,ESS_bulk,ESS_tail,ESS_bulk/s,R_hat
lp__,-21.239600,0.066915,2.228070,2.086550,-25.370100,-20.874700,-18.312500,1183.87,2013.180,196.852,1.00342
c1,0.005915,0.000051,0.001641,0.001119,0.002331,0.006411,0.007647,1400.42,739.906,232.859,1.00346
c2,0.001853,0.000053,0.001683,0.001226,0.000148,0.001340,0.005564,1390.35,750.354,231.185,1.00380
c3,0.004687,0.000058,0.001828,0.001760,0.001006,0.005115,0.006989,1050.03,1037.880,174.598,1.00040
c4,0.005633,0.000058,0.001867,0.001808,0.003289,0.005205,0.009366,1091.49,1279.730,181.491,1.00088
c5,0.019468,0.000245,0.011370,0.014664,0.001976,0.019166,0.037196,2044.44,1707.330,339.947,1.00314
c6,0.019030,0.000244,0.011357,0.014764,0.001345,0.019367,0.036450,2010.44,1506.830,334.293,1.00310
c7,0.006021,0.000018,0.000631,0.000673,0.004960,0.006028,0.007016,1292.47,2277.770,214.910,1.00005
f,0.358248,0.005629,0.179488,0.219378,0.077968,0.355326,0.643719,1027.98,1773.600,170.931,1.00115
a_11,0.027237,0.000245,0.011370,0.014730,0.009800,0.026978,0.044890,2043.91,1653.590,339.858,1.00290
